# Flagging Vendor invoices for manual review
**Objective:** Predict whether a vendor invoice should be flagged for manual approval based on abnormal cost, freight or delivery patterns, in order to reduce financial risk, improve efficiency, and prioritize human review where it adds the most value
- An automated flagging system enables finance teams to focus attention on high-risk invoices while low-risk invoices to be processed automatically.
  

In [ ]:
import pandas as pd
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
conn = sqlite3.connect("C:/Users/iamra/python+sql project/data/inventory.db")
tables = pd.read_sql_query("select name from sqlite_master where type= 'table'", conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print('Table name', table)
    df=pd.read_sql_query(f"select * from {table} limit 5", conn)
    display(df)

In [ ]:
df=pd.read_sql_query("""
WITH purchase_agg AS (
select
p.PONumber,
count(distinct p.brand) as total_brands,
sum(p.Quantity) as total_item_quantity,
sum(p.Dollars) as total_item_dollars,
avg(julianday(p.ReceivingDate) - julianday(p.PODate)) as avg_receiving_delay
from purchases p
group by p.PONumber
)

SELECT
vi.PONumber,
vi.Quantity as invoice_quantity,
vi.Dollars as invoice_dollars,
vi.Freight,
(julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
(julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
pa.total_brands,
pa.total_item_quantity,
pa.total_item_dollars,
pa.avg_receiving_delay

FROM vendor_invoice vi
LEFT JOIN purchase_agg pa
ON vi.PONumber=pa.PONumber
""", conn)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
def create_invoice_risk_label(row):
    #invoice total mismatch with item level total
    if (abs(row['invoice_dollars']- row["total_item_dollars"]) > 5):
        return 1
    #abnormality high receiving delay
    if row["avg_receiving_delay"]>10:
        return 1
    return 0

df["flag_invoice"]=df.apply(create_invoice_risk_label, axis=1)
df["flag_invoice"].value_counts()

In [ ]:
df["flag_invoice"].value_counts().plot(kind='bar')

In [ ]:
df.corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.iloc[:,1:-1].corr(), annot=True)
plt.show()

In [ ]:
flagged=df[df['flag_invoice']==1]
normal=df[df['flag_invoice']==0]

In [ ]:
significant_features=[]
non_significant_features=[]
results=[]


In [ ]:
metrics=['invoice_quantity', 'invoice_dollars', 'Freight', 
         'days_po_to_invoice', 'days_to_pay', 'total_brands', 'total_item_quantity', 'total_item_dollars', 'avg_receiving_delay']

In [ ]:
from scipy.stats import ttest_ind
for metric in metrics:
    flagged_mean=flagged[metric].mean()
    normal_mean=normal[metric].mean()

    t_stat, p_value= ttest_ind(
        flagged[metric].dropna(),
        normal[metric].dropna(),
        equal_var=False
    )
    if p_value<0.05:
        significant_features.append(metric)
        results.append({
            "metric": metric,
            "flagged_mean": flagged_mean.round(2),
            "normal_mean": normal_mean.round(2),
            "p_value": p_value.round(3)
        })
    else:
        non_significant_features.append(metric)
        print(metric)
        print({
            "metric": metric,
            "flagged_mean": flagged_mean.round(2),
            "normal_mean": normal_mean.round(2),
            "p_value": p_value.round(3)
        })
    

In [ ]:
non_significant_features

In [ ]:
significant_features

In [ ]:
results

In [ ]:
from sklearn.model_selection import train_test_split
X = df[['invoice_quantity','invoice_dollars','Freight','days_po_to_invoice','total_brands', 'total_item_quantity','total_item_dollars']]
y = df['flag_invoice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
X.describe().round()

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model1 = LogisticRegression(random_state = 42)
model1.fit(X_train_scaled, y_train)

model2 = DecisionTreeClassifier(random_state = 42)
model2.fit(X_train_scaled, y_train)

model3 = RandomForestClassifier(random_state = 42)
model3.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(model, X_test, y_test, model_name):
    
    predictions = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, predictions)
    
    print(f"\n{model_name} Performance:")
    print(f"Accuracy : {accuracy:.2f}")
    print("Classification Report :")
    print(classification_report(y_test, predictions))

In [ ]:
evaluate_model(model1, X_test_scaled, y_test, 'Logistic Regression')
evaluate_model(model2, X_test_scaled, y_test, 'Decision Tree Classifier')
evaluate_model(model3, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
model3.feature_importances_

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model3.feature_importances_
}).sort_values(by = "importance", ascending = False)

In [ ]:
feature_importance

In [ ]:
X = df[['invoice_quantity','invoice_dollars','Freight','total_item_quantity','total_item_dollars']]
y = df['flag_invoice']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model3 = RandomForestClassifier(random_state = 42)
model3.fit(X_train_scaled, y_train)

evaluate_model(model3, X_test_scaled, y_test, 'Random Forest Classifier')

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 4, 5, 6],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Model
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# F1 scorer
scorer = make_scorer(f1_score)

# Random Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

# Train
grid_search.fit(X_train_scaled, y_train)
evaluate_model(grid_search, X_test_scaled, y_test, 'Random Forest Classifier')


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(grid_search.predict(X_test_scaled), y_test)

In [ ]:
grid_search.best_params_